In [1]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkByExamples.com").getOrCreate()

In [3]:
from google.colab import files
uploaded=files.upload()


Saving carrier_master.csv to carrier_master.csv
Saving route_master.csv to route_master.csv
Saving shipments_75000.csv to shipments_75000.csv
Saving shipments_extra_12000.csv to shipments_extra_12000.csv
Saving warehouse_master.csv to warehouse_master.csv
Saving warehouse_sensor_config.json to warehouse_sensor_config.json


In [4]:
shipment_df=spark.read.csv("shipments_75000.csv",header=True,inferSchema=True)
shipment_df.show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000001|  2026-05-24|   2026-05-24|            2026-05-26| 

In [5]:
warehouse_df=spark.read.csv("warehouse_master.csv",header=True,inferSchema=True)
warehouse_df.show()

+------------+--------------------+-----------+------+--------------+--------------------+
|warehouse_id|      warehouse_name|       city|region|capacity_units|        storage_type|
+------------+--------------------+-----------+------+--------------+--------------------+
|         W01|Charminar Fulfilm...|  Hyderabad| South|         42000|             Ambient|
|         W02|  Electronic City DC|  Bengaluru| South|         55000|               Mixed|
|         W03|Vashi Distributio...|Navi Mumbai|  West|         61000|             Ambient|
|         W04|Okhla Sorting Centre|  New Delhi| North|         48000|               Mixed|
|         W05|Salt Lake Logisti...|    Kolkata|  East|         36000|             Ambient|
|         W06|Ambattur Dispatch...|    Chennai| South|         39000|Temperature Contr...|
|         W07|Naroda Distributi...|  Ahmedabad|  West|         44000|               Mixed|
|         W08|Hinjewadi Fulfilm...|       Pune|  West|         52000|             Ambient|

In [6]:
carrier_df=spark.read.csv("carrier_master.csv",header=True,inferSchema=True)
carrier_df.show()


+----------+--------------------+------------+----------------+-----------------+
|carrier_id|        carrier_name|carrier_type|base_cost_per_km|reliability_score|
+----------+--------------------+------------+----------------+-----------------+
|       C01|   SwiftLine Express|     Express|             6.4|               95|
|       C02|        ParcelBridge|    Standard|             4.3|               89|
|       C03|MetroRoute Logistics|    Standard|             4.7|               91|
|       C04|  AirSprint Couriers|     Express|             8.2|               94|
|       C05|           CargoNest|     Economy|             3.6|               84|
|       C06|           RouteWise|     Economy|             3.9|               86|
|       C07|   BluePeak Delivery|    Standard|             4.5|               90|
+----------+--------------------+------------+----------------+-----------------+



In [7]:
route_df=spark.read.csv("route_master.csv",header=True,inferSchema=True)
route_df.show()

+--------+-------------+------------------+-------------+-----------+--------+
|route_id|origin_region|destination_region|   route_type|distance_km|toll_inr|
+--------+-------------+------------------+-------------+-----------+--------+
|     R01|        South|             South|Intraregional|        180|     140|
|     R02|        South|              West|Interregional|        760|     490|
|     R03|        South|             North|Interregional|       1550|    1100|
|     R04|        South|              East|Interregional|       1430|     930|
|     R05|         West|              West|Intraregional|        230|     170|
|     R06|         West|             North|Interregional|       1130|     820|
|     R07|         West|              East|Interregional|       1710|    1250|
|     R08|        North|             North|Intraregional|        210|     150|
|     R09|        North|              East|Interregional|       1280|     910|
|     R10|         East|              East|Intraregi

In [8]:
shipment_df.printSchema()

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)



In [10]:
df=shipment_df.count()

In [11]:
print(df)

75000


In [12]:
shipment_df.show(15)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000001|  2026-05-24|   2026-05-24|            2026-05-26| 

In [14]:
df=shipment_df.select(
    "shipment_id",
    "warehouse_id",
    "service_level",
    "shipment_status",
    "shipping_fee_inr"
).show()

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
| SHP0000011|         W04|     Standar

In [17]:
from pyspark.sql.functions import col
df=shipment_df.filter(
    (col("service_level")=="Express") &
    (col("package_weight_kg") > 5)
).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

In [18]:
df=shipment_df.sort(
    col("shipping_fee_inr").desc()
).show(20)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0044904|  2026-08-11|   2026-08-13|            2026-08-16| 

In [19]:
from pyspark.sql.functions import count

df=shipment_df.groupBy("shipment_status") \
    .agg(count("*").alias("total_shipments")) \
    .show()


+---------------+---------------+
|shipment_status|total_shipments|
+---------------+---------------+
|       Returned|           6130|
|           Lost|           2241|
|     In Transit|          10472|
|      Cancelled|           3723|
|      Delivered|          52434|
+---------------+---------------+



In [21]:
from pyspark.sql.functions import sum, col

df=shipment_df.fillna({"shipping_cost_inr": 0}) \
    .groupBy("service_level") \
    .agg(
        sum("shipping_fee_inr").alias("total_shipping_fees"),
        sum("shipping_cost_inr").alias("total_shipping_costs")
    ) \
    .withColumn(
        "profit_margin",
        col("total_shipping_fees") - col("total_shipping_costs")
    ) \
    .show()

+-------------+--------------------+--------------------+------------------+
|service_level| total_shipping_fees|total_shipping_costs|     profit_margin|
+-------------+--------------------+--------------------+------------------+
|      Express|1.0058468059999986E7|   7350805.699999983| 2707662.360000002|
|     Standard|1.3437372619999977E7|1.0110465900000025E7|3326906.7199999522|
|      Economy|   4489918.680000002|   3386212.500000007|1103706.1799999946|
+-------------+--------------------+--------------------+------------------+



In [22]:
from pyspark.sql.functions import count

df=shipment_df.join(
    warehouse_df,
    "warehouse_id",
    "inner"
  ).groupBy("region") \
 .agg(count("*").alias("total_shipments")) \
 .show()

+------+---------------+
|region|total_shipments|
+------+---------------+
| South|          33018|
|  East|           5306|
|  West|          26909|
| North|           9767|
+------+---------------+



In [23]:
from pyspark.sql.functions import avg, col

df=shipment_df.join(
    carrier_df,
    "carrier_id"
  ).join(
    route_df,
    "route_id"
  ).filter(
    col("shipping_cost_inr").isNotNull() &
    (col("distance_km") > 0)
  ).withColumn(
    "cost_per_km",
    col("shipping_cost_inr") / col("distance_km")
  ).groupBy("carrier_name") \
 .agg(
     avg("cost_per_km").alias("avg_cost_per_km")
 ) \
 .show()

+--------------------+------------------+
|        carrier_name|   avg_cost_per_km|
+--------------------+------------------+
|MetroRoute Logistics| 0.594275960971091|
|  AirSprint Couriers|0.6125755103088726|
|           CargoNest|0.5897283901696387|
|   BluePeak Delivery|0.6074832283099014|
|        ParcelBridge|0.5957409571811839|
|           RouteWise|0.5879324060720517|
|   SwiftLine Express|0.6032748764541479|
+--------------------+------------------+



In [24]:
from pyspark.sql.functions import col, sum, when

df=shipment_df.select(
    sum(when(col("carrier_id").isNull(), 1).otherwise(0)).alias("carrier_id_nulls"),
    sum(when(col("package_weight_kg").isNull(), 1).otherwise(0)).alias("package_weight_nulls"),
    sum(when(col("shipping_cost_inr").isNull(), 1).otherwise(0)).alias("shipping_cost_nulls"),
    sum(when(col("customer_rating").isNull(), 1).otherwise(0)).alias("customer_rating_nulls")
).show()

+----------------+--------------------+-------------------+---------------------+
|carrier_id_nulls|package_weight_nulls|shipping_cost_nulls|customer_rating_nulls|
+----------------+--------------------+-------------------+---------------------+
|             871|                1129|               1503|                26255|
+----------------+--------------------+-------------------+---------------------+



In [25]:
shipment_handling_df = shipment_df.fillna(
    {"handling_minutes": 0}
)

shipment_handling_df.show()

shipment_no_null_carrier_df = shipment_df.filter(
    col("carrier_id").isNotNull()
)

shipment_no_null_carrier_df.show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000001|  2026-05-24|   2026-05-24|            2026-05-26| 

In [26]:
from pyspark.sql.functions import when, col

df = shipment_df.withColumn(
    "weight_category",
    when(col("package_weight_kg").isNull(), "Unknown")
    .when(col("package_weight_kg") < 2, "Light")
    .when(
        (col("package_weight_kg") >= 2) &
        (col("package_weight_kg") <= 10),
        "Medium"
    )
    .otherwise("Heavy")
)

df.show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+---------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|weight_category|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+---------------+
| SHP0000001|  2

In [27]:
from pyspark.sql.functions import col, to_date, datediff

shipment_df = shipment_df \
    .withColumn("booking_date", to_date(col("booking_date"), "yyyy-MM-dd")) \
    .withColumn("promised_delivery_date", to_date(col("promised_delivery_date"), "yyyy-MM-dd")) \
    .withColumn("actual_delivery_date", to_date(col("actual_delivery_date"), "yyyy-MM-dd"))

delivered_df = shipment_df.filter(
    col("shipment_status") == "Delivered"
).withColumn(
    "delivery_duration_days",
    datediff(
        col("actual_delivery_date"),
        col("booking_date")
    )
)

delivered_df.show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+----------------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|delivery_duration_days|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------------------

In [28]:
from pyspark.sql.functions import when, col

shipment_df = shipment_df.withColumn(
    "delivery_performance",
    when(
        col("actual_delivery_date").isNull(),
        "No Delivery Date"
    )
    .when(
        col("actual_delivery_date") < col("promised_delivery_date"),
        "Early"
    )
    .when(
        col("actual_delivery_date") == col("promised_delivery_date"),
        "On Time"
    )
    .otherwise("Late")
)

shipment_df.select(
    "shipment_id",
    "promised_delivery_date",
    "actual_delivery_date",
    "delivery_performance"
).show()

+-----------+----------------------+--------------------+--------------------+
|shipment_id|promised_delivery_date|actual_delivery_date|delivery_performance|
+-----------+----------------------+--------------------+--------------------+
| SHP0000001|            2026-05-26|          2026-05-27|                Late|
| SHP0000002|            2026-05-21|          2026-05-21|             On Time|
| SHP0000003|            2026-02-03|          2026-02-04|                Late|
| SHP0000004|            2026-03-17|          2026-03-18|                Late|
| SHP0000005|            2026-08-25|          2026-08-29|                Late|
| SHP0000006|            2026-02-09|          2026-02-09|             On Time|
| SHP0000007|            2026-01-14|                NULL|    No Delivery Date|
| SHP0000008|            2026-03-16|                NULL|    No Delivery Date|
| SHP0000009|            2026-08-18|          2026-08-21|                Late|
| SHP0000010|            2026-06-10|          2026-0

In [29]:
from pyspark.sql.functions import year, month, count, sum

shipment_df.filter(
    col("shipment_status") == "Delivered"
).groupBy(
    year("booking_date").alias("year"),
    month("booking_date").alias("month")
).agg(
    count("*").alias("delivered_shipments"),
    sum("shipping_fee_inr").alias("total_shipping_fees")
).orderBy(
    "year",
    "month"
).show()


+----+-----+-------------------+-------------------+
|year|month|delivered_shipments|total_shipping_fees|
+----+-----+-------------------+-------------------+
|2026|    1|               6710|  2525138.440000005|
|2026|    2|               6097| 2268113.5000000037|
|2026|    3|               6810| 2530456.2400000026|
|2026|    4|               6379| 2389163.4600000014|
|2026|    5|               6858| 2549683.9999999963|
|2026|    6|               6615| 2447590.1200000006|
|2026|    7|               6773| 2512355.2199999997|
|2026|    8|               6192| 2306432.5199999996|
+----+-----+-------------------+-------------------+



In [30]:
from pyspark.sql.functions import sum, dense_rank
from pyspark.sql.window import Window

carrier_fees = shipment_df.join(
    warehouse_df, "warehouse_id"
).join(
    carrier_df, "carrier_id"
).groupBy(
    "region",
    "carrier_name"
).agg(
    sum("shipping_fee_inr").alias("total_shipping_fees")
)

window_spec = Window.partitionBy("region") \
                    .orderBy(col("total_shipping_fees").desc())

carrier_fees.withColumn(
    "rank",
    dense_rank().over(window_spec)
).show()

+------+--------------------+-------------------+----+
|region|        carrier_name|total_shipping_fees|rank|
+------+--------------------+-------------------+----+
|  East|   SwiftLine Express| 393862.61999999994|   1|
|  East|        ParcelBridge|          373590.02|   2|
|  East|MetroRoute Logistics| 302260.92000000004|   3|
|  East|           CargoNest| 243322.91999999998|   4|
|  East|   BluePeak Delivery| 222025.85999999993|   5|
|  East|  AirSprint Couriers| 211583.64000000004|   6|
|  East|           RouteWise|  195810.3399999999|   7|
| North|        ParcelBridge|  731442.0999999994|   1|
| North|   SwiftLine Express|  653223.6200000005|   2|
| North|MetroRoute Logistics|  597883.4600000005|   3|
| North|           CargoNest| 464601.85999999975|   4|
| North|   BluePeak Delivery| 425396.69999999995|   5|
| North|  AirSprint Couriers| 393181.07999999984|   6|
| North|           RouteWise| 338296.45999999985|   7|
| South|        ParcelBridge| 2442652.5400000014|   1|
| South|  

In [31]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_spec = Window.partitionBy("warehouse_id") \
                    .orderBy(
                        col("booking_date").desc(),
                        col("shipment_id").desc()
                    )

latest_shipments = shipment_df.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
)

latest_shipments.show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+---+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|delivery_performance| rn|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+---------------

In [32]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

window_spec = Window.partitionBy("warehouse_id") \
                    .orderBy(
                        "booking_date",
                        "shipment_id"
                    ) \
                    .rowsBetween(
                        Window.unboundedPreceding,
                        Window.currentRow
                    )

shipment_df.withColumn(
    "running_total_shipping_fee",
    sum("shipping_fee_inr").over(window_spec)
).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+--------------------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|delivery_performance|running_total_shipping_fee|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+

In [35]:
extra_df = spark.read.csv("shipments_extra_12000.csv", header=True, inferSchema=True)
combined_df = shipment_df.unionByName(extra_df, allowMissingColumns=True)

combined_df.show()

print("Total Records =", combined_df.count())

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|delivery_performance|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+--------------------+
|

In [37]:
from pyspark.sql.functions import count

shipment_df.groupBy("shipment_id") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1") \
    .show()

+-----------+---+
|shipment_id|cnt|
+-----------+---+
+-----------+---+



In [38]:
before_count = shipment_df.count()

dedup_df = shipment_df.dropDuplicates(["shipment_id"])

after_count = dedup_df.count()

print("Before Deduplication:", before_count)
print("After Deduplication :", after_count)
print("Duplicates Removed  :", before_count - after_count)

Before Deduplication: 75000
After Deduplication : 75000
Duplicates Removed  : 0


In [39]:
from pyspark.sql.functions import col, sum

pipeline_df = shipment_df.filter(
    col("shipment_status") == "Delivered"
).join(
    warehouse_df,
    "warehouse_id"
).withColumn(
    "profit",
    col("shipping_fee_inr") - col("shipping_cost_inr")
).groupBy(
    "region"
).agg(
    sum("profit").alias("total_profit")
).orderBy(
    col("total_profit").desc()
)

# Inspect Logical & Physical Plan
pipeline_df.explain()

# Trigger Action
pipeline_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_profit#1815 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_profit#1815 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1805]
      +- HashAggregate(keys=[region#139], functions=[sum(profit#1814)])
         +- Exchange hashpartitioning(region#139, 200), ENSURE_REQUIREMENTS, [plan_id=1802]
            +- HashAggregate(keys=[region#139], functions=[partial_sum(profit#1814)])
               +- Project [region#139, (shipping_fee_inr#28 - shipping_cost_inr#29) AS profit#1814]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#136], Inner, BuildRight, false
                     :- Project [warehouse_id#22, shipping_fee_inr#28, shipping_cost_inr#29]
                     :  +- Filter ((isnotnull(shipment_status#30) AND (shipment_status#30 = Delivered)) AND isnotnull(warehouse_id#22))
                     :     +- FileScan csv [warehouse_id#22,shipping_fee_inr#28,shipping_cost_inr#2

In [40]:
import time
from pyspark.sql.functions import sum

delivered_df = shipment_df.filter(
    col("shipment_status") == "Delivered"
).cache()

# First count()
start = time.time()
delivered_df.count()
print("First count():", time.time() - start)

# Second count()
start = time.time()
delivered_df.count()
print("Second count():", time.time() - start)

# First aggregation
start = time.time()
delivered_df.agg(
    sum("shipping_fee_inr")
).show()
print("First aggregation:", time.time() - start)

# Second aggregation
start = time.time()
delivered_df.agg(
    sum("shipping_fee_inr")
).show()
print("Second aggregation:", time.time() - start)

First count(): 4.67331600189209
Second count(): 0.14969348907470703
+---------------------+
|sum(shipping_fee_inr)|
+---------------------+
| 1.9528933499999907E7|
+---------------------+

First aggregation: 0.5205588340759277
+---------------------+
|sum(shipping_fee_inr)|
+---------------------+
| 1.9528933499999907E7|
+---------------------+

Second aggregation: 0.16497015953063965


In [41]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.functions import sum, avg, col

express_df = shipment_df.filter(
    col("service_level") == "Express"
).persist(StorageLevel.MEMORY_AND_DISK)

express_df.count()
express_df.agg(
    sum("shipping_fee_inr").alias("total_fee"),
    avg("shipping_fee_inr").alias("avg_fee")
).show()

print(express_df.storageLevel)

express_df.unpersist()

print("Data Released")

+--------------------+------------------+
|           total_fee|           avg_fee|
+--------------------+------------------+
|1.0058468059999986E7|476.93068089141707|
+--------------------+------------------+

Disk Memory Serialized 1x Replicated
Data Released


In [42]:
from pyspark.sql.functions import spark_partition_id, count

repartitioned_df = shipment_df.repartition(8)

print("Partitions:", repartitioned_df.rdd.getNumPartitions())

repartitioned_df.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy(
    "partition_id"
).agg(
    count("*").alias("record_count")
).orderBy(
    "partition_id"
).show()

Partitions: 8
+------------+------------+
|partition_id|record_count|
+------------+------------+
|           0|        9375|
|           1|        9374|
|           2|        9376|
|           3|        9375|
|           4|        9375|
|           5|        9375|
|           6|        9375|
|           7|        9375|
+------------+------------+



In [43]:
df12 = shipment_df.repartition(12)

print(df12.rdd.getNumPartitions())

coalesce_df = df12.coalesce(3)

print(coalesce_df.rdd.getNumPartitions())

repartition_df = df12.repartition(3)

print(repartition_df.rdd.getNumPartitions())

12
3
3


In [44]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

#2
avg_weight = shipment_df.select(
    avg("package_weight_kg")
).first()[0]

clean_df = shipment_df.fillna({
    "shipping_cost_inr": 0
}).fillna({
    "package_weight_kg": avg_weight
})

#3
clean_df = clean_df \
    .withColumn(
        "booking_date",
        to_date("booking_date")
    ) \
    .withColumn(
        "promised_delivery_date",
        to_date("promised_delivery_date")
    ) \
    .withColumn(
        "actual_delivery_date",
        to_date("actual_delivery_date")
    )

    #4
joined_df = clean_df \
    .join(warehouse_df, "warehouse_id", "left") \
    .join(carrier_df, "carrier_id", "left") \
    .join(route_df, "route_id", "left")

    #5
enriched_df = joined_df \
    .withColumn(
        "delivery_duration_days",
        datediff(
            col("actual_delivery_date"),
            col("booking_date")
        )
    ) \
    .withColumn(
        "late_delivery",
        when(
            col("actual_delivery_date") >
            col("promised_delivery_date"),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "on_time_delivery",
        when(
            col("actual_delivery_date") <=
            col("promised_delivery_date"),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "shipment_profit",
        col("shipping_fee_inr") -
        col("shipping_cost_inr")
    )

#6
agg_df = enriched_df \
    .groupBy(
        "region",
        "carrier_name"
    ) \
    .agg(
        sum("shipping_fee_inr")
            .alias("total_revenue"),

        sum("shipment_profit")
            .alias("total_profit"),

        count("*")
            .alias("shipment_count"),

        sum("late_delivery")
            .alias("late_shipments"),

        sum("on_time_delivery")
            .alias("on_time_shipments")
    ) \
    .withColumn(
        "late_delivery_pct",
        round(
            col("late_shipments") * 100 /
            col("shipment_count"),
            2
        )
    ) \
    .withColumn(
        "on_time_delivery_pct",
        round(
            col("on_time_shipments") * 100 /
            col("shipment_count"),
            2
        )
    )

#7
rank_window = Window \
    .partitionBy("region") \
    .orderBy(
        col("on_time_delivery_pct").desc()
    )

final_report = agg_df.withColumn(
    "carrier_rank",
    dense_rank().over(rank_window)
)

#8
final_report.cache()

#9
final_report.show()

final_report.orderBy(
    col("total_profit").desc()
).show()

final_report.groupBy(
    "region"
).agg(
    sum("total_profit").alias("region_profit")
).show()

#10
final_report \
    .repartition(4) \
    .write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(
        "final_logistics_report.csv"
    )

+------+--------------------+------------------+------------------+--------------+--------------+-----------------+-----------------+--------------------+------------+
|region|        carrier_name|     total_revenue|      total_profit|shipment_count|late_shipments|on_time_shipments|late_delivery_pct|on_time_delivery_pct|carrier_rank|
+------+--------------------+------------------+------------------+--------------+--------------+-----------------+-----------------+--------------------+------------+
| South|           CargoNest|1437480.9200000009| 364710.4200000005|          3838|          1084|             1656|            28.24|               43.15|           1|
| South|MetroRoute Logistics|1929008.1199999978| 494540.6199999999|          5197|          1544|             2157|            29.71|                41.5|           2|
| South|           RouteWise|1233214.2399999998|         316221.74|          3328|           987|             1373|            29.66|               41.26|      